# recfold: explore recursive calls

Run the cells from top to bottom in JupyterLab, Notebook, or a compatible JupyterHub.
Install this repository into the active kernel first, for example with
`%pip install -e /path/on/server/to/recfold`. Use the branch containing the renderer.
No extra visualization packages or network assets are needed.

The examples use small inputs so both the call tree and folded graph stay readable.


In [ ]:
from recfold.tracer import trace, run_traced
from recfold.renderer import render_session


## 1. Fibonacci: find repeated work

Switch between **Folded DAG** and **Call tree**. Select `fib(2)` in the folded graph
and change **Recorded occurrence** to inspect its different original calls.
Expected: 15 recorded calls, 6 unique subtrees, depth 4, 60% repeated calls.
Folding summarizes execution; it does not memoize or speed up this function.


In [ ]:
@trace
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(5) =", fib(5))
fib_widget = render_session(fib.last_session, title="Fibonacci: repeated work", height=620)
fib_widget


## 2. A plain recursive function

`run_traced` returns the function result and its trace. Factorial makes a chain,
so there is no repeated subtree to fold in this example.


In [ ]:
def factorial(n):
    return 1 if n <= 1 else n * factorial(n - 1)

answer, factorial_session = run_traced(factorial, 5)
print("factorial(5) =", answer)
render_session(factorial_session, title="Factorial: a chain of calls")


## 3. Backtracking: inspect a mutable path at call entry

This function pushes a digit, explores a branch, then pops the digit again.
The recorded input copies keep each call's entry path even after it is mutated.
Select nodes in the **Call tree** and compare **Arguments** and **Entry snapshot**.

This is entry-state inspection, not a debugger stepping through every append/pop.
The tracer does not automatically capture arbitrary local variables.


In [ ]:
solutions = []

@trace
def binary_paths(path, remaining):
    if remaining == 0:
        solutions.append(path.copy())
        return path.copy()
    for digit in (0, 1):
        path.append(digit)
        binary_paths(path, remaining - 1)
        path.pop()

path = []
binary_paths(path, 2)
print("Solutions:", solutions)
print("Original path after backtracking:", path)
render_session(binary_paths.last_session, title="Backtracking: saved path inputs", height=620)


## 4. Inspect a partial trace after a guard stops recursion

The guard rejects the next call before creating its node. Already-entered calls
remain available, and their exit snapshots show the propagated error.
Red-outlined nodes have recorded errors. Select one to read its details.


In [ ]:
@trace(max_depth=3)
def countdown(n):
    return 0 if n == 0 else countdown(n - 1)

try:
    countdown(10)
except RecursionError as error:
    print(error)

render_session(countdown.last_session, title="Depth guard: partial trace")


## 5. Export an interactive file

Uncomment the next line to save the Fibonacci visualization in the current
folder. Open it in a regular browser; it works offline. The export contains the
recorded inputs/results. Existing files are preserved unless `overwrite=True`.


In [ ]:
# fib_widget.save("fib.html")
# fib_widget.save("fib.html", overwrite=True)  # intentionally replace a prior export


## Controls and troubleshooting

- Click a node, or Tab to it and press Enter/Space.
- Drag the graph background to pan. Use +/− or the toolbar to zoom, 0 to fit,
  arrow keys to pan, and Ctrl/⌘ + scroll to zoom.
- Each output is isolated; changing one widget does not change the others.
- Trust and rerun this notebook if scripts were disabled. A managed Hub may
  restrict iframe output; in that case use the standalone HTML export.
- See [the renderer guide](renderer.md) for API details and current limitations.
